In [ ]:
import csv
import json
from pathlib import Path
import shutil

seed_digit = 1
target_os = "Android"
target_behavior_class = (seed_digit % 5) + 1

data_dir = Path("data")
output_dir = Path("output")
backup_dir = Path("backup")

input_file = data_dir / "user_behavior_dataset.csv"
filtered_csv = output_dir / "Jerusalem_TUPM-26-1131_mobile_filtered.csv"
summary_json = output_dir / "Jerusalem_TUPM-26-1131_mobile_summary.json"
backup_csv = backup_dir / "Jerusalem_TUPM-26-1131_mobile_filtered_backup.csv"

total_records = 0
os_matches = 0
behavior_matches = 0
selected_users = []
total_battery = 0.0
total_data = 0.0

with open(input_file, mode="r", encoding="utf-8", errors="ignore") as f:
    reader = csv.DictReader(f)
    for row in reader:
        total_records += 1
        os = row.get("Operating System", row.get("os", ""))
        behavior_class = row.get("User Behavior Class", row.get("behavior_class", ""))
        
        if os.strip().lower() == target_os.lower():
            os_matches += 1
            if str(behavior_class).strip() == str(target_behavior_class):
                behavior_matches += 1
                
                user_id = row.get("User ID", row.get("user_id", ""))
                device_model = row.get("Device Model", row.get("device_model", ""))
                battery_drain = float(row.get("Battery Drain (mAh/day)", row.get("battery_drain", 0)))
                data_usage = float(row.get("Data Usage (MB/day)", row.get("data_usage", 0)))
                
                total_battery += battery_drain
                total_data += data_usage
                
                selected_users.append({
                    "User ID": user_id,
                    "Device Model": device_model,
                    "Operating System": os,
                    "Battery Drain (mAh/day)": battery_drain,
                    "Data Usage (MB/day)": data_usage
                })

avg_battery = (total_battery / behavior_matches) if behavior_matches > 0 else 0.0
avg_data = (total_data / behavior_matches) if behavior_matches > 0 else 0.0

fieldnames = ["User ID", "Device Model", "Operating System", "Battery Drain (mAh/day)", "Data Usage (MB/day)"]
with open(filtered_csv, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(selected_users)

summary_data = {
    "operating_system": target_os,
    "behavior_class": target_behavior_class,
    "total_records_in_dataset": total_records,
    "records_matching_os": os_matches,
    "records_in_target_behavior": behavior_matches,
    "average_battery_drain": round(avg_battery, 2),
    "average_data_usage": round(avg_data, 2),
    "exported_row_count": len(selected_users)
}

with open(summary_json, mode="w", encoding="utf-8") as f:
    json.dump(summary_data, f, indent=4)

shutil.copy(filtered_csv, backup_csv)
backup_size_bytes = backup_csv.stat().st_size

print(f"Mobile Profiling Completed. Exported Users: {len(selected_users)}")
print(f"Backup File Size: {backup_size_bytes} bytes")

In [ ]:
import csv
import json
from pathlib import Path
import shutilgit push origin activity6-additional-exercises
from collections import Counter

seed_digit = 1
target_year = 2015 + seed_digit
git add .
data_dir = Path("data")
output_dir = Path("output")
backup_dir = Path("backup")

input_file = data_dir / "Global_Cybersecurity_Threats_2015-2024.csv"
filtered_csv = output_dir / "Jerusalem_TUPM-26-1131_cyber_filtered.csv"
summary_json = output_dir / "Jerusalem_TUPM-26-1131_cyber_summary.json"
backup_json = backup_dir / "Jerusalem_TUPM-26-1131_cyber_summary_backup.json"
report_file = output_dir / "Jerusalem_TUPM-26-1131_cyber_report.txt"

total_records = 0
target_incidents = []
attack_counts = Counter()
total_loss = 0.0

with open(input_file, mode="r", encoding="utf-8", errors="ignore") as f:
    reader = csv.DictReader(f)
    for row in reader:
        total_records += 1
        date_str = row.get("Date", row.get("Year", row.get("date", "")))
        
        if str(target_year) in date_str:
            attack_type = row.get("Attack Type", row.get("attack_type", "Unknown"))
            try:
                loss = float(row.get("Financial Loss (in Million $)", row.get("loss", 0)))
            except ValueError:
                loss = 0.0
                
            attack_counts[attack_type] += 1
            total_loss += loss
            
            target_incidents.append({
                "Incident ID": row.get("Incident ID", row.get("id", str(total_records))),
                "Date": date_str,
                "Attack Type": attack_type,
                "Financial Loss (in Million $)": loss
            })

incident_count = len(target_incidents)
avg_loss = (total_loss / incident_count) if incident_count > 0 else 0.0

sorted_incidents = sorted(target_incidents, key=lambda x: x["Financial Loss (in Million $)"], reverse=True)
top_5_costliest = sorted_incidents[:5]

fieldnames = ["Incident ID", "Date", "Attack Type", "Financial Loss (in Million $)"]
with open(filtered_csv, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(target_incidents)

summary_data = {
    "target_year": target_year,
    "total_records_in_dataset": total_records,
    "incidents_in_target_year": incident_count,
    "count_per_attack_type": dict(attack_counts),
    "total_financial_loss_million_usd": round(total_loss, 2),
    "average_financial_loss_million_usd": round(avg_loss, 2),
    "five_costliest_incidents": top_5_costliest
}

with open(summary_json, mode="w", encoding="utf-8") as f:
    json.dump(summary_data, f, indent=4)

shutil.copy(summary_json, backup_json)
backup_verified = backup_json.exists()

with open(report_file, mode="w", encoding="utf-8") as f:
    f.write(f"CYBERSECURITY INCIDENT REPORT ({target_year})\n")
    f.write(f"Student: Jerusalem (TUPM-26-1131)\n")
    f.write(f"Total Incidents: {incident_count}\n")
    f.write(f"Total Financial Loss: ${total_loss:,.2f} Million\n")
    f.write(f"Average Financial Loss: ${avg_loss:,.2f} Million\n\n")
    f.write("Top 5 Costliest Incidents:\n")
    for inc in top_5_costliest:
        f.write(f"- ID: {inc['Incident ID']} | Type: {inc['Attack Type']} | Loss: ${inc['Financial Loss (in Million $)']}M\n")

print(f"Cybersecurity Report Completed for Year {target_year}.")
print(f"Total Incidents Found: {incident_count}, Backup Verified: {backup_verified}")